# Fake News Detection — TF-IDF + Logistic Regression
### Detectron AI · Module 08

This notebook trains a **TF-IDF + Logistic Regression** text classifier to distinguish
credible-sounding text from clickbait/misleading text, and compares it against the
transparent linguistic-feature scorer (clickbait phrases, emotional language, sourcing)
used in the live web demo.


In [1]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt


## 1. Labeled training corpus (misleading=1, credible=0)

In [2]:
data = [
    ("SHOCKING: doctors hate this one trick they don't want you to know!!!", 1),
    ("You won't believe what happens next, this goes viral instantly", 1),
    ("BREAKING: secret cure BANNED by big pharma, exposed at last", 1),
    ("This miracle remedy destroys all your health problems instantly", 1),
    ("Scientists SHOCKED by this unbelievable discovery, must see now", 1),
    ("Government doesn't want you to see this terrifying truth", 1),
    ("This one weird trick will change your life forever guaranteed", 1),
    ("Celebrity SLAMS rival in explosive interview everyone is talking about", 1),
    ("You won't believe this shocking secret they tried to hide", 1),
    ("BANNED video reveals the truth they don't want exposed", 1),
    ("This proves everyone was wrong about the vaccine forever", 1),
    ("Doctors are FURIOUS about this new method going viral", 1),
    ("According to a study published in Nature, researchers found a modest correlation.", 0),
    ("The central bank confirmed an interest rate increase of 0.25 percent today.", 0),
    ("Officials said the new policy will take effect at the start of next quarter.", 0),
    ("Data from the national statistics office shows employment rose slightly in May.", 0),
    ("The research team published their findings in a peer-reviewed journal yesterday.", 0),
    ("City council confirmed the budget allocation following a public hearing.", 0),
    ("Researchers at the university found a measurable decline in emissions this year.", 0),
    ("The company's quarterly earnings report showed steady growth according to filings.", 0),
    ("Local authorities released an official statement clarifying the new regulations.", 0),
    ("A peer-reviewed clinical trial reported moderate improvement in patient outcomes.", 0),
    ("The transportation department confirmed the bridge closure for scheduled maintenance.", 0),
    ("Census data indicates a gradual shift in regional population over the decade.", 0),
]

df = pd.DataFrame(data, columns=["text", "label"])
df["label_name"] = df["label"].map({1: "misleading", 0: "credible"})
df


,text,label,label_name
0,SHOCKING: doctors hate this one trick they don...,1,misleading
1,"You won't believe what happens next, this goes...",1,misleading
2,"BREAKING: secret cure BANNED by big pharma, ex...",1,misleading
3,This miracle remedy destroys all your health p...,1,misleading
4,Scientists SHOCKED by this unbelievable discov...,1,misleading
5,Government doesn't want you to see this terrif...,1,misleading
6,This one weird trick will change your life for...,1,misleading
7,Celebrity SLAMS rival in explosive interview e...,1,misleading
8,You won't believe this shocking secret they tr...,1,misleading
9,BANNED video reveals the truth they don't want...,1,misleading


## 2. TF-IDF vectorization + train/test split

In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    df["text"], df["label"], test_size=0.3, random_state=42, stratify=df["label"]
)

vectorizer = TfidfVectorizer(ngram_range=(1,2), min_df=1)
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)
print("Feature count:", X_train_vec.shape[1])


Feature count: 265


## 3. Train Logistic Regression

In [4]:
model = LogisticRegression(max_iter=1000, class_weight="balanced")
model.fit(X_train_vec, y_train)
pred = model.predict(X_test_vec)

print(classification_report(y_test, pred, target_names=["credible", "misleading"]))


              precision    recall  f1-score   support

    credible       0.80      1.00      0.89         4
  misleading       1.00      0.75      0.86         4

    accuracy                           0.88         8
   macro avg       0.90      0.88      0.87         8
weighted avg       0.90      0.88      0.87         8



## 4. Most predictive n-grams

In [5]:
feature_names = np.array(vectorizer.get_feature_names_out())
coefs = model.coef_[0]
top_misleading = feature_names[np.argsort(coefs)[-12:][::-1]]
top_credible = feature_names[np.argsort(coefs)[:12]]

print("Top phrases -> misleading:")
print(top_misleading)
print()
print("Top phrases -> credible:")
print(top_credible)


Top phrases -> misleading:
['this' 'they' 'you' 'forever' 'your' 'instantly' 'you won' 'won believe'
 'believe' 'won' 'don' 'don want']

Top phrases -> credible:
['the' 'in' 'confirmed' 'confirmed the' 'an' 'new' 'the new' 'of'
 'following public' 'following' 'the budget' 'public hearing']


## 5. Transparent linguistic feature scorer (matches the live web demo)

In [6]:
CLICKBAIT = ["you won't believe", "shocking", "what happens next", "doctors hate",
             "this one trick", "secret", "exposed", "miracle cure", "banned", "goes viral", "breaking"]
EMOTIONAL = ["outrage", "fury", "terrifying", "devastating", "explosive", "stunning", "bombshell"]
CREDIBILITY_MARKERS = ["according to", "study published", "researchers found", "data shows",
                       "official statement", "confirmed by", "peer-reviewed"]

def analyze_fake_news(text):
    lower = text.lower()
    score = 0
    signals = []
    clickbait_hits = sum(p in lower for p in CLICKBAIT)
    if clickbait_hits:
        score += clickbait_hits * 15
        signals.append(f"Clickbait phrasing x{clickbait_hits}")
    exclam = text.count("!")
    if exclam >= 2:
        score += 10
        signals.append("Excessive exclamation marks")
    emo_hits = sum(p in lower for p in EMOTIONAL)
    if emo_hits:
        score += emo_hits * 10
        signals.append(f"Emotional language x{emo_hits}")
    cred_hits = sum(p in lower for p in CREDIBILITY_MARKERS)
    if cred_hits == 0:
        score += 12
        signals.append("No cited sources")
    else:
        score -= cred_hits * 8
        signals.append(f"Source attribution present x{cred_hits}")
    score = max(0, min(100, score))
    verdict = "likely misleading" if score >= 55 else "needs verification" if score >= 30 else "likely credible"
    return score, verdict, signals

score, verdict, signals = analyze_fake_news("SHOCKING: doctors hate this secret, exposed at last!!!")
print(f"Score: {score}/100 -> {verdict.upper()}")
for s in signals: print(" -", s)


Score: 82/100 -> LIKELY MISLEADING
 - Clickbait phrasing x4
 - Excessive exclamation marks
 - No cited sources


## Notes

- TF-IDF + Logistic Regression captures statistical word patterns from labeled examples,
  while the rule-based scorer encodes known stylistic markers of misinformation directly —
  useful when you don't have a large labeled training set yet.
- Production fake-news detectors (e.g. research systems built on the **LIAR** or
  **FakeNewsNet** datasets) typically combine linguistic features like these with
  source-credibility databases and cross-referencing against fact-checking APIs.
